In [12]:
# ----------------------------######----------------------------#
#   _idchk_0901_pkls_GET_report                               #
# ----------------------------######----------------------------#

import os
import pandas as pd
from tqdm import tqdm


def _idchk_0901_pkls_GET_report(
    root_folder,
    id_col="ID",
    recursive=True,
    return_df_all=False,
    return_df_unique=False,
    show_examples=10,
):
    """
    READ-ONLY ID health check for a folder of .pkl DataFrames.

    What it answers:
      - Are IDs unique (among valid IDs)?
      - How many rows are missing IDs?
      - Does row count == "real ID entries" (rows with valid IDs)?
      - Which PKLs contain missing IDs or duplicates?

    Parameters
    ----------
    root_folder : str
      folder containing PKLs
    id_col : str
      ID column name (default "ID")
    recursive : bool
      scan subfolders
    return_df_all : bool
      if True, returns concatenated df_all
    return_df_unique : bool
      if True, returns df_unique (deduped by ID, valid IDs only)
    show_examples : int
      print small samples for missing/duplicates (0 disables)

    Returns
    -------
    report : dict
      contains counts + per-file summaries (+ optional dfs)
    """

    if not os.path.isdir(root_folder):
        raise NotADirectoryError(root_folder)

    # ---- collect PKLs ----
    pkl_files = []
    if recursive:
        for dirpath, _, filenames in os.walk(root_folder):
            for f in filenames:
                if f.lower().endswith(".pkl"):
                    pkl_files.append(os.path.join(dirpath, f))
    else:
        for f in os.listdir(root_folder):
            if f.lower().endswith(".pkl"):
                pkl_files.append(os.path.join(root_folder, f))

    if not pkl_files:
        print("⚠️ No PKL files found.")
        return {"ok": False, "reason": "no_pkls_found"}

    # ---- load PKLs ----
    dfs = []
    load_errors = []

    for fp in tqdm(pkl_files, desc="Loading PKLs (ID check)"):
        try:
            obj = pd.read_pickle(fp)
            if not isinstance(obj, pd.DataFrame):
                continue

            if obj.empty:
                # skip empty DFs to avoid pandas concat FutureWarning
                continue

            df = obj.copy()
            df["_src_pkl"] = os.path.basename(fp)
            df["_src_path"] = fp
            dfs.append(df)

        except Exception as e:
            load_errors.append({"file": fp, "error": str(e)})

    if not dfs:
        print("⚠️ No valid non-empty DataFrames loaded.")
        return {"ok": False, "reason": "no_valid_dfs", "load_errors": load_errors}

    # ---- concat (safe; empties already removed) ----
    df_all = pd.concat(dfs, ignore_index=True, sort=False)

    if id_col not in df_all.columns:
        raise ValueError(f"❌ Column '{id_col}' not found in concatenated data.")

    # ---- define "valid ID" ----
    s = df_all[id_col]
    # valid if not NaN and not empty string after strip
    valid_mask = (~s.isna()) & (s.astype(str).str.strip() != "")
    missing_mask = ~valid_mask

    df_valid = df_all.loc[valid_mask].copy()

    # ---- core counts ----
    n_total_rows = len(df_all)
    n_valid_rows = int(valid_mask.sum())
    n_missing_rows = int(missing_mask.sum())

    n_unique_ids = int(df_valid[id_col].nunique(dropna=True))
    n_dup_rows = int(df_valid.duplicated(subset=[id_col], keep=False).sum())

    ids_strictly_unique = (n_valid_rows == n_unique_ids) and (n_dup_rows == 0)

    # ---- per-file summaries ----
    per_file = (
        df_all.assign(_missing_id=missing_mask)
        .groupby("_src_pkl", dropna=False)
        .agg(
            rows=("__dummy__", "size") if "__dummy__" in df_all.columns else (id_col, "size"),
            missing_id=("_missing_id", "sum"),
        )
        .reset_index()
    )
    # the above "rows" calc can be brittle if id_col missing in some rows; safer:
    per_file["rows"] = df_all.groupby("_src_pkl").size().values

    # duplicates per file (only meaningful for valid IDs)
    dup_mask_valid = df_valid.duplicated(subset=[id_col], keep=False)
    per_file_dup = (
        df_valid.loc[dup_mask_valid]
        .groupby("_src_pkl")
        .size()
        .reset_index(name="dup_rows")
    )
    per_file = per_file.merge(per_file_dup, on="_src_pkl", how="left")
    per_file["dup_rows"] = per_file["dup_rows"].fillna(0).astype(int)

    # ---- optional examples ----
    df_missing_example = pd.DataFrame()
    df_dups_example = pd.DataFrame()

    if show_examples and n_missing_rows > 0:
        df_missing_example = df_all.loc[missing_mask].head(show_examples)

    if show_examples and n_dup_rows > 0:
        df_dups_example = df_valid.loc[dup_mask_valid].sort_values(by=id_col).head(show_examples)

    # ---- print report ----
    print("\n==================== ID HEALTH REPORT ====================")
    print(f"PKLs scanned (found)        : {len(pkl_files)}")
    print(f"PKLs loaded (non-empty DF)  : {len(dfs)}")
    print(f"Load errors                 : {len(load_errors)}")
    print("----------------------------------------------------------")
    print(f"Total rows (all)            : {n_total_rows}")
    print(f"Valid ID rows               : {n_valid_rows}")
    print(f"Missing ID rows             : {n_missing_rows}")
    print("----------------------------------------------------------")
    print(f"Unique IDs (valid only)     : {n_unique_ids}")
    print(f"Duplicate-ID rows (valid)   : {n_dup_rows}")
    print("----------------------------------------------------------")
    print(f"IDs strictly unique?        : {ids_strictly_unique}")
    print("==========================================================")

    if show_examples and n_missing_rows > 0:
        print("\n⚠️ Example rows with MISSING ID:")
        cols_show = ["_src_pkl", id_col] + [c for c in df_all.columns if c not in ["_src_pkl", "_src_path", id_col]][:6]
        print(df_missing_example[cols_show])

    if show_examples and n_dup_rows > 0:
        print("\n⚠️ Example rows with DUPLICATE IDs:")
        cols_show = ["_src_pkl", id_col] + [c for c in df_all.columns if c not in ["_src_pkl", "_src_path", id_col]][:6]
        print(df_dups_example[cols_show])

    # ---- df_unique (valid IDs only) ----
    df_unique = pd.DataFrame()
    if return_df_unique:
        df_unique = df_valid.drop_duplicates(subset=[id_col], keep="first").copy()

    report = {
        "ok": True,
        "root_folder": root_folder,
        "id_col": id_col,
        "pkls_found": len(pkl_files),
        "pkls_loaded_nonempty_df": len(dfs),
        "load_errors": load_errors,
        "total_rows": n_total_rows,
        "valid_id_rows": n_valid_rows,
        "missing_id_rows": n_missing_rows,
        "unique_ids_valid": n_unique_ids,
        "duplicate_id_rows_valid": n_dup_rows,
        "ids_strictly_unique": ids_strictly_unique,
        "per_file_summary": per_file.sort_values(["missing_id", "dup_rows"], ascending=False),
    }

    if show_examples:
        report["missing_id_examples"] = df_missing_example
        report["duplicate_id_examples"] = df_dups_example

    if return_df_all:
        report["df_all"] = df_all

    if return_df_unique:
        report["df_unique"] = df_unique

    return report


In [22]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!
root_folder = "mp3s"


report = _idchk_0901_pkls_GET_report(
    root_folder,
    id_col="ID",
    recursive=True,
    return_df_all=False,     # keep False for check-only
    return_df_unique=False,  # keep False for check-only
    show_examples=10
)

# If you want to see which PKLs are problematic:
df_per_file = report["per_file_summary"]
df_per_file


Loading PKLs (ID check): 100%|█████████████████████████████████████████████████| 5/5 [00:00<00:00, 102.90it/s]


==================== ID HEALTH REPORT ====================
PKLs scanned (found)        : 5
PKLs loaded (non-empty DF)  : 5
Load errors                 : 0
----------------------------------------------------------
Total rows (all)            : 1139
Valid ID rows               : 1139
Missing ID rows             : 0
----------------------------------------------------------
Unique IDs (valid only)     : 1139
Duplicate-ID rows (valid)   : 0
----------------------------------------------------------
IDs strictly unique?        : True



/var/folders/pp/n6z2gh0x56ngpvp6vx5ml9j00000gn/T/ipykernel_70829/3357675157.py:94: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat(dfs, ignore_index=True, sort=False)


,_src_pkl,rows,missing_id,dup_rows
0,df_000_.pkl,188,0,0
1,df_001_.pkl,258,0,0
2,df_002_.pkl,182,0,0
3,df_003_.pkl,283,0,0
4,df_USAt_final.pkl,228,0,0


In [14]:
# -----######-----###### AIFF COUNTER (Recursive iCloud-safe) -----######-----###### #
import os
from tqdm import tqdm

def _aiff_2908_icld_count_GET_total(root_folder, audio_extensions=None, include_placeholders=False):
    """
    Input:
      - root_folder: path to scan (str)
      - audio_extensions: list-like of extensions to count (e.g., ['.aiff', '.aif']); case-insensitive. If None → ['.aiff','.aif']
      - include_placeholders: if True, also count iCloud placeholders (*.aiff.icloud / *.aif.icloud)
    Output:
      - result dict:
          {
            'count_real': int,          # *.aiff / *.aif (not *.icloud)
            'count_placeholders': int,  # *.aiff.icloud / *.aif.icloud
            'count_total': int,         # real + (placeholders if include_placeholders)
            'scanned_files': int,       # all filenames encountered
            'scanned_dirs': int         # directories walked
          }
    Notes:
      - Skips macOS junk '._*' and '.DS_Store'
      - Matching is case-insensitive
    """

    # ---------- setup ----------
    if audio_extensions is None:
        audio_extensions = ['.aiff', '.aif']

    # Normalize and prep sets
    exts = tuple(e.lower() for e in audio_extensions)

    count_real = 0
    count_placeholders = 0
    scanned_files = 0
    scanned_dirs = 0

    # ---------- walk with TQM ----------
    # First collect all dirs to build a smooth TQM
    all_dirs = []
    for dpath, _, _ in os.walk(root_folder):
        all_dirs.append(dpath)

    for dpath in tqdm(all_dirs, desc="Scanning folders (AIFF counter)"):
        scanned_dirs += 1
        try:
            entries = os.listdir(dpath)
        except Exception:
            continue

        for name in entries:
            # Skip macOS junk
            if name.startswith('._') or name in ('.DS_Store',):
                continue

            scanned_files += 1
            low = name.lower()

            # Identify placeholders: *.ext.icloud
            is_placeholder = low.endswith('.icloud') and any(low.endswith(ext + '.icloud') for ext in exts)

            # Identify real: ends with ext and NOT .icloud
            is_real = (any(low.endswith(ext) for ext in exts) and not low.endswith('.icloud'))

            if is_real:
                count_real += 1
            elif is_placeholder:
                count_placeholders += 1

    count_total = count_real + (count_placeholders if include_placeholders else 0)

    # Summary print (so you get instant readout)
    print("===== AIFF COUNT SUMMARY =====")
    print(f"Root: {root_folder}")
    print(f"Real AIFFs (.aiff/.aif):     {count_real}")
    print(f"iCloud placeholders:         {count_placeholders}")
    print(f"TOTAL (reported):            {count_total}   {'(includes placeholders)' if include_placeholders else '(excludes placeholders)'}")
    print(f"Dirs scanned:                {scanned_dirs}")
    print(f"Files seen:                  {scanned_files}")
    print("================================")

    return {
        'count_real': count_real,
        'count_placeholders': count_placeholders,
        'count_total': count_total,
        'scanned_files': scanned_files,
        'scanned_dirs': scanned_dirs
    }


# Icloud

In [15]:
root_folder = "/Users/yerik/Library/Mobile Documents/com~apple~CloudDocs/_YDB_/_2_ms_icloud/_1_NEW_SOURCE"

# Count only real AIFF files (exclude .icloud placeholders)
res = _aiff_2908_icld_count_GET_total(
    root_folder=root_folder,
    audio_extensions=['.aiff', '.aif'],
    include_placeholders=False
)

# If you also want to include iCloud placeholders in the reported total, flip this to True:
# res = _aiff_2908_icld_count_GET_total(root_folder, ['.aiff','.aif'], include_placeholders=True)


Scanning folders (AIFF counter): 100%|█████████████████████████████████████| 78/78 [00:00<00:00, 26438.96it/s]

===== AIFF COUNT SUMMARY =====
Root: /Users/yerik/Library/Mobile Documents/com~apple~CloudDocs/_YDB_/_2_ms_icloud/_1_NEW_SOURCE
Real AIFFs (.aiff/.aif):     1916
iCloud placeholders:         0
TOTAL (reported):            1916   (excludes placeholders)
Dirs scanned:                78
Files seen:                  1993


# LOCAL

In [16]:
root_folder = "/Users/yerik/Music/_1_NEW_SOURCE"

# Count only real AIFF files (exclude .icloud placeholders)
res = _aiff_2908_icld_count_GET_total(
    root_folder=root_folder,
    audio_extensions=['.aiff', '.aif'],
    include_placeholders=False
)

# If you also want to include iCloud placeholders in the reported total, flip this to True:
# res = _aiff_2908_icld_count_GET_total(root_folder, ['.aiff','.aif'], include_placeholders=True)


Scanning folders (AIFF counter): 100%|███████████████████████████████████| 159/159 [00:00<00:00, 31838.74it/s]

===== AIFF COUNT SUMMARY =====
Root: /Users/yerik/Music/_1_NEW_SOURCE
Real AIFFs (.aiff/.aif):     3339
iCloud placeholders:         0
TOTAL (reported):            3339   (excludes placeholders)
Dirs scanned:                159
Files seen:                  3544


# compare both folders 

In [17]:
# -----######-----###### CORE IMPORTABLE FUNCTION (Compare AIFF/WAV/etc. Files Between Two Folders) -----######-----###### #
import os
from pathlib import Path
import pandas as pd
from tqdm import tqdm

def _compare_0109_folders_GET_df_diff(
    folder_a,
    folder_b,
    exts=None
):
    """
    Compare file names in two folders (recursively).
    Returns a DataFrame with files only in A or only in B.
    
    Params:
      folder_a (str): First folder path.
      folder_b (str): Second folder path.
      exts (list or None): Optional list of extensions to filter (e.g., [".aiff", ".aif", ".wav"])
    """
    folder_a, folder_b = Path(folder_a), Path(folder_b)
    exts = [e.lower() for e in exts] if exts else None

    def collect_files(folder):
        files = []
        for f in tqdm(folder.rglob("*"), desc=f"Scanning {folder.name}"):
            if f.is_file():
                if exts is None or f.suffix.lower() in exts:
                    files.append(f.name)
        return set(files)

    set_a = collect_files(folder_a)
    set_b = collect_files(folder_b)

    only_a = sorted(set_a - set_b)
    only_b = sorted(set_b - set_a)

    df_diff = pd.DataFrame({
        "only_in_A": only_a + [""] * (max(len(only_a), len(only_b)) - len(only_a)),
        "only_in_B": only_b + [""] * (max(len(only_a), len(only_b)) - len(only_b)),
    })

    return df_diff


In [18]:
root_a = "/Users/yerik/Music/_1_NEW_SOURCE"
root_b = "/Users/yerik/Library/Mobile Documents/com~apple~CloudDocs/_YDB_/_2_ms_icloud/_1_NEW_SOURCE"

df_diff = _compare_0109_folders_GET_df_diff(root_a, root_b, exts=[".aiff", ".aif", ".wav", ".mp3"])
print(df_diff.head(30))  # show first 30 rows


Scanning _1_NEW_SOURCE: 3582it [00:00, 114513.92it/s]
Scanning _1_NEW_SOURCE: 2020it [00:00, 46771.12it/s]

                         only_in_A only_in_B
0    001_MINIMAL_DEEP_TECH_124.mp3          
1    002_MINIMAL_DEEP_TECH_127.mp3          
2   003_MINIMAL_DEEP_HOUSE_122.mp3          
3   004_MINIMAL_DEEP_HOUSE_124.mp3          
4    005_MINIMAL_DEEPHOUSE_126.mp3          
5               006_HOUSE_122_.mp3          
6               007_HOUSE_124_.mp3          
7               008_HOUSE_125_.mp3          
8              009_HOUSE_125_2.mp3          
9               010_HOUSE_126_.mp3          
10              011_HOUSE_127_.mp3          
11             012_HOUSE_127_2.mp3          
12              013_HOUSE_128_.mp3          
13              014_HOUSE_129_.mp3          
14              015_HOUSE_130_.mp3          
15              016_HOUSE_131_.mp3          
16          017_DEEP_HOUSE_122.mp3          
17          018_DEEP_HOUSE_123.mp3          
18          019_DEEP_HOUSE_124.mp3          
19          020_DEEP_HOUSE_125.mp3          
20          021_DEEP_HOUSE_126.mp3          
21        

# next set the whole thing about the right paths checinkg if filee exit trhough ID, and quality code

# check an ID

In [19]:
# -----######-----###### CHECK & DISPLAY ROW FOR ID -----######-----###### #
target_id = "t6-5802"

df_match = df_unique[df_unique["ID"].astype(str) == target_id]

if df_match.empty:
    print(f"❌ '{target_id}' not found in df_unique['ID']")
else:
    print(f"✅ '{target_id}' FOUND — {len(df_match)} row(s)")
    print(df_match)


NameError: name 'df_unique' is not defined

# sinc ones 

In [10]:
# -----######-----###### CONCAT MULTIPLE PKL FILES INTO ONE DF -----######-----###### #
import pandas as pd
from pathlib import Path

def _concat_2608_pkls_GET_df(folder, pkl_files):
    """
    Input:
      - folder: string, path to the directory containing .pkl files
      - pkl_files: list of .pkl filenames to concat
    Output:
      - Concatenated DataFrame
    """
    dfs = []
    for f in pkl_files:
        path = Path(folder) / f
        try:
            df = pd.read_pickle(path)
            dfs.append(df)
        except Exception as e:
            print(f"⚠️ Could not load {f}: {e}")
    
    if not dfs:
        raise ValueError("No DataFrames loaded.")
    
    df_concat = pd.concat(dfs, ignore_index=True)
    return df_concat


#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!
folder = "SINC"  # 🔴 change to your folder path
pkl_files =''# ["df_sin1t_final.pkl", "df_sinc2t_final.pkl", "df_sinc3t_final.pkl"]

df_final = _concat_2608_pkls_GET_df(folder, pkl_files)

print(df_final.shape)
#print(df_final.head())


ValueError: No DataFrames loaded.

In [18]:
import pandas as pd

# show all columns
pd.set_option("display.max_columns", None)

# show all rows (if you also want every row)
# pd.set_option("display.max_rows", None)

#print(df_final.head())   # or just df_final


In [19]:
# Check uniqueness of the 'ID' column
is_unique = df_final['ID'].is_unique
print("Are all IDs unique? ➜", is_unique)

# If not unique, see how many duplicates
if not is_unique:
    dup_count = df_final['ID'].duplicated().sum()
    print(f"⚠️ Found {dup_count} duplicate IDs")

    # Show the actual duplicate IDs
    print(df_final['ID'][df_final['ID'].duplicated()].unique())


Are all IDs unique? ➜ True


### now in folder 

In [20]:
root_folder = "/Users/yerik/Music/_3_YODJ_ADDS-inbox"

# Count only real AIFF files (exclude .icloud placeholders)
res = _aiff_2908_icld_count_GET_total(
    root_folder=root_folder,
    audio_extensions=['.mp3', '.mp3'],
    include_placeholders=False
)

# If you also want to include iCloud placeholders in the reported total, flip this to True:
# res = _aiff_2908_icld_count_GET_total(root_folder, ['.aiff','.aif'], include_placeholders=True)


Scanning folders (AIFF counter): 100%|██████████████████████████████████████| 27/27 [00:00<00:00, 8932.50it/s]

===== AIFF COUNT SUMMARY =====
Root: /Users/yerik/Music/_3_YODJ_ADDS-inbox
Real AIFFs (.aiff/.aif):     1261
iCloud placeholders:         0
TOTAL (reported):            1261   (excludes placeholders)
Dirs scanned:                27
Files seen:                  1288
